In [ ]:
# Cell 1: Imports and setup
from common_imports import *
from common_utils import evaluate, train_model, setup_data
from torch.optim.lr_scheduler import StepLR

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

# --- 1. 定义残差块 (BasicBlock) ---
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        # 如果尺寸不匹配，通过 1x1 卷积对齐
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x) # 残差相加
        return torch.relu(out)

# --- 2. 组装终极模型 ---
class FashionResNet(nn.Module):
    def __init__(self):
        super(FashionResNet, self).__init__()
        self.in_planes = 64
        
        # 初始特征提取
        self.prep = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # 典型的 ResNet 三阶段
        self.layer1 = BasicBlock(64, 64, stride=1)
        self.layer2 = BasicBlock(64, 128, stride=2)  # 14x14
        self.layer3 = BasicBlock(128, 256, stride=2) # 7x7
        
        # 全局池化层替代了冗余的全连接层
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, 10)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)
    
# --- 3. 增强版数据增强 (包含随机擦除) ---
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)) # 迫使模型学习局部细节
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# --- 4. 优化器与损失函数 (启用标签平滑) ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = FashionResNet().to(device)

# Label Smoothing: 0.1 能有效解决分类界限模糊的问题
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

# CosineAnnealing: 让模型在最后阶段精准收敛
scheduler = CosineAnnealingLR(optimizer, T_max=50)

# 3) Data setup with augmentation
train_transform = transforms.Compose([
    
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    # transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3)),
    transforms.Normalize((0.1307), (0.3081)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307), (0.3081)),
])

# 4) Training the model
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

# --- 4. 优化器与损失函数 (启用标签平滑) ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = FashionResNet().to(device)

# Label Smoothing: 0.1 能有效解决分类界限模糊的问题
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

# CosineAnnealing: 让模型在最后阶段精准收敛
scheduler = CosineAnnealingLR(optimizer, T_max=50)

full_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
val_size = int(len(full_train_dataset) * 0.1)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=test_transform), batch_size=128, shuffle=False, num_workers=0, pin_memory=False)




print("=== Training the model ===")
history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=NUM_EPOCHS,
    patience=4,
)

# 5) Evaluate the model
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\nFinal Test Loss: {test_loss:.4f} | Final Test Accuracy: {test_acc:.4f}")

=== Training the model ===
Epoch 01/15 | Train Loss: 0.7594 | Train Acc: 0.7905 | Val Loss: 0.3784 | Val Acc: 0.8597
Epoch 02/15 | Train Loss: 0.3889 | Train Acc: 0.8569 | Val Loss: 0.3147 | Val Acc: 0.8838
Epoch 03/15 | Train Loss: 0.3437 | Train Acc: 0.8731 | Val Loss: 0.3128 | Val Acc: 0.8852
Epoch 04/15 | Train Loss: 0.3125 | Train Acc: 0.8845 | Val Loss: 0.2731 | Val Acc: 0.8968
Epoch 05/15 | Train Loss: 0.2918 | Train Acc: 0.8922 | Val Loss: 0.2665 | Val Acc: 0.9047
Epoch 06/15 | Train Loss: 0.2752 | Train Acc: 0.8989 | Val Loss: 0.2422 | Val Acc: 0.9110
Epoch 07/15 | Train Loss: 0.2628 | Train Acc: 0.9027 | Val Loss: 0.2422 | Val Acc: 0.9110
